# 04. Custom Aggregation Functions in Pandas

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week10/04.Custom-Aggregation-Functions/notebooks/01_04.Custom-Aggregation-Functions.ipynb)

## Overview
While built-in statistical functions like `.mean()` and `.sum()` cover standard needs, data analysts frequently require **domain-specific metrics**—such as pass rates, interquartile ranges, or service-level compliance percentages.

In this notebook, we learn how to:
1. **Author Custom Aggregators**: Write Python functions that accept a `pd.Series` and return a scalar value.
2. **Assign Custom Column Names**: Use tuples `('Column_Name', function)` inside `.agg()`.
3. **Use Inline Lambdas**: Implement lightweight calculations on the fly.
4. **Handle Missing Data Safely**: Clean groups before computing statistical metrics.

## 1. Setup: Student Unit Marks Dataset

We create a dataset of assessment marks across three introductory computing units.

In [ ]:
import pandas as pd
import numpy as np

df_marks = pd.DataFrame({
    'unit': [
        'ITEC102', 'ITEC102', 'ITEC102', 'ITEC102', 'ITEC102',
        'ITEC105', 'ITEC105', 'ITEC105', 'ITEC105', 'ITEC105',
        'ITEC108', 'ITEC108', 'ITEC108', 'ITEC108', 'ITEC108'
    ],
    'student_id': list(range(101, 116)),
    'mark': [
        88, 42, 95, 76, 68,
        55, 62, 48, 70, 81,
        91, 85, 88, 79, 94
    ]
})

print("Student Marks Dataset:")
display(df_marks.head(10))

## 2. Defining Custom Aggregation Functions

A custom aggregation function must:
1. Accept a single `pd.Series` parameter representing the values in each group.
2. Return a single scalar number (int, float, str, or bool).
3. Handle edge cases (e.g. empty groups or missing values) gracefully.

In [ ]:
def data_range(s: pd.Series) -> float:
    """Computes the statistical range (max - min)."""
    cleaned = s.dropna()
    return cleaned.max() - cleaned.min() if not cleaned.empty else np.nan

def interquartile_range(s: pd.Series) -> float:
    """Computes the Interquartile Range (Q3 - Q1), resistant to outliers."""
    cleaned = s.dropna()
    return cleaned.quantile(0.75) - cleaned.quantile(0.25) if len(cleaned) >= 2 else np.nan

def pass_rate_pct(s: pd.Series) -> float:
    """Calculates the percentage of students who passed (mark >= 50)."""
    cleaned = s.dropna()
    return (cleaned >= 50).mean() * 100.0 if not cleaned.empty else 0.0

def trimmed_mean(s: pd.Series) -> float:
    """Computes a trimmed mean excluding the lowest and highest mark."""
    cleaned = s.dropna().sort_values()
    return cleaned.iloc[1:-1].mean() if len(cleaned) > 2 else cleaned.mean()

## 3. Applying Custom Aggregations with `.agg()`

Pass a list of `('Column_Header', function)` tuples to apply your custom functions and provide clear column names.

In [ ]:
custom_summary = df_marks.groupby('unit')['mark'].agg([
    ('Cohort_Size', 'count'),
    ('Mean_Mark', 'mean'),
    ('Range', data_range),
    ('IQR', interquartile_range),
    ('Trimmed_Mean', trimmed_mean),
    ('Pass_Rate_pct', pass_rate_pct)
])

print("Custom Unit Assessment Metrics:")
display(custom_summary.round(1))

## 4. Lightweight Aggregation with Lambdas

For short, one-line calculations, Python's anonymous `lambda` functions provide a fast alternative to defining a named function.

In [ ]:
lambda_results = df_marks.groupby('unit')['mark'].agg([
    ('Avg_Mark', 'mean'),
    ('Distinction_Rate_pct', lambda s: (s >= 75).mean() * 100.0)
])

print("Distinction Rates (Marks >= 75):")
display(lambda_results.round(1))

## 5. Practical Exercises

### Exercise: Call Centre Service Level and Rating Spread
Below is customer call data across three support centres.
1. Define a custom function `sla_compliance(s)` calculating the percentage of calls handled within 180 seconds (`s <= 180`).
2. Define a custom function `rating_range(s)` calculating the difference between the highest and lowest rating (`max - min`).
3. Group by `centre` and compute both metrics alongside `total_calls`.

In [ ]:
calls = pd.DataFrame({
    'centre': ['Sydney', 'Sydney', 'Sydney', 'Melbourne', 'Melbourne', 'Melbourne', 'Brisbane', 'Brisbane'],
    'shift': ['Morning', 'Evening', 'Morning', 'Morning', 'Evening', 'Evening', 'Morning', 'Evening'],
    'handling_time_sec': [120, 240, 150, 180, 95, 310, 140, 200],
    'rating': [4.8, 3.2, 4.5, 4.0, 4.9, 2.8, 4.2, 3.9]
})

# --- Student Code Here ---
# def sla_compliance(s):
# def rating_range(s):
# results = ...

# --- Solution ---
def sla_compliance(s: pd.Series) -> float:
    return (s <= 180).mean() * 100.0

def rating_range(s: pd.Series) -> float:
    return s.max() - s.min()

results = calls.groupby('centre').agg(
    total_calls=('handling_time_sec', 'count'),
    sla_compliance_pct=('handling_time_sec', sla_compliance),
    rating_spread=('rating', rating_range)
)
display(results.round(1))

## 6. Key Takeaways

1. **Input Signature**: Custom aggregators must take a `pd.Series` and return a scalar value.
2. **Robustness**: Always handle missing data using `.dropna()` before calculating metrics like range or quantile.
3. **Lambdas for Simplicity**: Use lambdas for quick boolean percentage calculations (e.g. `lambda s: (s > threshold).mean() * 100`).
4. **Tuple Labelling**: Use `('Column_Name', func)` to assign descriptive names to custom aggregates.